In [1]:
# !pip install geopy

In [2]:
import pandas as pd
import numpy as np
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

### Stočių ir matavimų duomenys

In [3]:
station = pd.read_csv("Station.csv")
station = station.drop(["_type", "_revision", "_page.next"], axis=1)
station.head()

,_id,stat_num,latitude,longitude
0,cb2cef5d-4776-4341-941d-07ad0a398b0b,1,54.677625,25.285186
1,6bd59444-5437-447d-bc2d-c32ef5b0a964,2,54.686111,25.210835
2,ed74723a-e2dd-41c4-a3b0-ae7c2cb0695c,3,54.715278,25.289444
3,53f89320-7ac7-4c90-9612-95b663284fb7,4,54.673333,25.248903
4,d3a305f2-183b-4d20-8b28-f2e71f885ad6,6,NaN,NaN


Iš koordinačių ištraukiami stočių miestai.

In [4]:
geolocator = Nominatim(user_agent="geo_project")
reverse = RateLimiter(geolocator.reverse, min_delay_seconds=1)

def get_city(lat, lon):
    if pd.isna(lat) or pd.isna(lon):
        return np.nan

    location = reverse((lat, lon), language="en")

    if location is None:
        return np.nan

    address = location.raw['address']

    return (
        address.get('city') or
        address.get('town') or
        address.get('village') or
        address.get('municipality') or
        address.get('county') or
        np.nan
    )

station['city'] = station.apply(lambda x: get_city(x['latitude'], x['longitude']), axis=1)

In [5]:
station

,_id,stat_num,latitude,longitude,city
0,cb2cef5d-4776-4341-941d-07ad0a398b0b,1,54.677625,25.285186,Vilnius
1,6bd59444-5437-447d-bc2d-c32ef5b0a964,2,54.686111,25.210835,Vilnius
2,ed74723a-e2dd-41c4-a3b0-ae7c2cb0695c,3,54.715278,25.289444,Vilnius
3,53f89320-7ac7-4c90-9612-95b663284fb7,4,54.673333,25.248903,Vilnius
4,d3a305f2-183b-4d20-8b28-f2e71f885ad6,6,NaN,NaN,NaN
5,27cbf611-ff8f-4851-bf66-e87d96303ad8,11,NaN,NaN,NaN
6,8628cd14-cc2c-4b7a-a17d-98e9b9d42083,12,55.725000,24.365569,Panevėžys
7,f39d3d59-6a18-4d34-954a-d39e4c8f86a5,21,56.319440,22.870840,Naujoji Akmenė
8,37d8b588-c538-4db7-be57-dfd1527c6588,22,55.937833,23.308028,Šiauliai
9,c533d7e0-d443-48e6-b123-6da13ca87db6,23,56.309722,22.331389,Mažeikiai


In [6]:
averages = pd.read_csv("Averages.csv")
averages = averages.drop(["_type", "_revision", "_page.next"], axis=1)
averages.head()

C:\Users\rugil\AppData\Local\Temp\ipykernel_31492\391105750.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  averages = pd.read_csv("Averages.csv")


,_id,id,stat_num._id,ldatetime,code_combi,lvalue,atribut
0,0883565b-c0b7-490d-8687-f73ed09a5bc6,100000004,6bd59444-5437-447d-bc2d-c32ef5b0a964,2012-12-11T10:00:00,6201,0.00,1
1,78ed962e-9b5e-4b2e-91e1-af211488a26b,100000005,6bd59444-5437-447d-bc2d-c32ef5b0a964,2012-12-11T10:00:00,6901,0.00,1
2,dbb449c9-84c9-40e2-b7e8-56fe24d7043d,100000009,6bd59444-5437-447d-bc2d-c32ef5b0a964,2012-12-11T10:00:00,4501,1.03,1
3,5e33d873-cbd7-4715-b8e5-0769f65dd321,100000010,6bd59444-5437-447d-bc2d-c32ef5b0a964,2012-12-11T10:00:00,7101,0.01,1
4,703a188a-f59b-496c-accf-75c38b13c74c,100000011,6bd59444-5437-447d-bc2d-c32ef5b0a964,2012-12-11T10:00:00,5201,0.30,1


Stotys apjungiamos su pagrindine matavymų lentele.

In [7]:
averages_stations = averages.merge(
    station[['_id', 'stat_num', 'city']], 
    left_on='stat_num._id', 
    right_on='_id', 
    how='left'
)

averages_stations = averages_stations.drop(["stat_num._id", "_id_y"], axis=1)
averages_stations.head()

,_id_x,id,ldatetime,code_combi,lvalue,atribut,stat_num,city
0,0883565b-c0b7-490d-8687-f73ed09a5bc6,100000004,2012-12-11T10:00:00,6201,0.00,1,2,Vilnius
1,78ed962e-9b5e-4b2e-91e1-af211488a26b,100000005,2012-12-11T10:00:00,6901,0.00,1,2,Vilnius
2,dbb449c9-84c9-40e2-b7e8-56fe24d7043d,100000009,2012-12-11T10:00:00,4501,1.03,1,2,Vilnius
3,5e33d873-cbd7-4715-b8e5-0769f65dd321,100000010,2012-12-11T10:00:00,7101,0.01,1,2,Vilnius
4,703a188a-f59b-496c-accf-75c38b13c74c,100000011,2012-12-11T10:00:00,5201,0.30,1,2,Vilnius


In [8]:
missing_cities = averages_stations['city'].isna().sum()
print(f"Eilutės be miesto: {missing_cities}")

if missing_cities > 0:
    print(averages_stations[averages_stations['city'].isna()]['stat_num'].unique())


Eilutės be miesto: 37660
[99 11 32  6]


### Taršos ir matavimo vienetų duomenys

In [9]:
units= pd.read_csv("Units.csv")
units = units.drop(["_type", "_revision", "_page.next"], axis=1)
units

,_id,code_unit,unitname,id
0,28f650e6-08d2-4311-98fe-55cc2aba9ff0,1,ppb,1
1,48518a60-6e96-4b6f-93b8-bcaeff67685a,2,ppm,2
2,89a0ce55-265d-4266-91ea-5eb0ed25b77b,3,ug/m3,3
3,0fc540c4-cae8-4557-a091-6e310dd8d095,4,m/s,4
4,85e37543-6901-475a-83e1-0647bbd4b2bc,5,deg,5
5,67461e7c-506e-430a-8962-cd25ebed54da,6,hPa,6
6,93b8a619-9e3b-4204-8957-4dee6a075d4a,7,^C,7
7,866e2570-2343-40fa-b0e7-999c75f9a46e,8,%,8
8,4797ee11-9fda-4178-930f-13ad3ae435a6,9,W/m2,9
9,2b1b3b00-a4d3-4ea4-9dff-92374f6fc07b,10,kod,10


In [10]:
quantity = pd.read_csv("Quantity.csv")
quantity = quantity.drop(["_type", "_revision", "_page.next"], axis=1)
quantity.head()

,_id,code_quantity,shortname,longname,attrib,code_unit._id,id
0,0b30092e-0bac-4a8a-a21b-ea3916ae6e85,1,SO2,sulphur dioxide,gasep,28f650e6-08d2-4311-98fe-55cc2aba9ff0,1
1,686bb9d9-e770-4034-b6d9-f75a6d922141,2,NO,nitrogen monoxide,gasep,28f650e6-08d2-4311-98fe-55cc2aba9ff0,2
2,ad7c3c3e-8cd7-4f15-b89b-669e28660296,3,NO2,nitrogen dioxide,gasep,28f650e6-08d2-4311-98fe-55cc2aba9ff0,3
3,f04d89f4-440d-4b6b-b7c2-0331245f6940,4,NOx,nitrogen oxides,gasep,28f650e6-08d2-4311-98fe-55cc2aba9ff0,4
4,1fefab1d-1525-41a8-8a03-d82b041f4dbc,5,PM10,PM10 particulates,parti,89a0ce55-265d-4266-91ea-5eb0ed25b77b,5


In [11]:
quantity_units = pd.read_csv("QuantityUnits.csv")
quantity_units = quantity_units.drop(["_type", "_revision", "_page.next"], axis=1)
quantity_units.head()

,_id,code_combi,code_quantity._id,code_unit._id
0,dcc66f7a-4802-4f7d-b9bc-f790137c8613,101,0b30092e-0bac-4a8a-a21b-ea3916ae6e85,28f650e6-08d2-4311-98fe-55cc2aba9ff0
1,02643e24-3205-4af1-a608-76009dacff59,102,0b30092e-0bac-4a8a-a21b-ea3916ae6e85,48518a60-6e96-4b6f-93b8-bcaeff67685a
2,d55fe2d8-d1dc-44aa-8433-b520480bd5d7,103,0b30092e-0bac-4a8a-a21b-ea3916ae6e85,89a0ce55-265d-4266-91ea-5eb0ed25b77b
3,24f92792-dc63-4b2c-a715-69410ebd1f02,104,0b30092e-0bac-4a8a-a21b-ea3916ae6e85,0fc540c4-cae8-4557-a091-6e310dd8d095
4,f6e05409-ede2-45a9-b5d8-5d260f28af48,105,0b30092e-0bac-4a8a-a21b-ea3916ae6e85,85e37543-6901-475a-83e1-0647bbd4b2bc


In [12]:
full_units = pd.merge(
    quantity_units, 
    units[['_id', 'code_unit', 'unitname']], 
    left_on='code_unit._id', 
    right_on='_id', 
    how='left'
)

full_units.head()

,_id_x,code_combi,code_quantity._id,code_unit._id,_id_y,code_unit,unitname
0,dcc66f7a-4802-4f7d-b9bc-f790137c8613,101,0b30092e-0bac-4a8a-a21b-ea3916ae6e85,28f650e6-08d2-4311-98fe-55cc2aba9ff0,28f650e6-08d2-4311-98fe-55cc2aba9ff0,1,ppb
1,02643e24-3205-4af1-a608-76009dacff59,102,0b30092e-0bac-4a8a-a21b-ea3916ae6e85,48518a60-6e96-4b6f-93b8-bcaeff67685a,48518a60-6e96-4b6f-93b8-bcaeff67685a,2,ppm
2,d55fe2d8-d1dc-44aa-8433-b520480bd5d7,103,0b30092e-0bac-4a8a-a21b-ea3916ae6e85,89a0ce55-265d-4266-91ea-5eb0ed25b77b,89a0ce55-265d-4266-91ea-5eb0ed25b77b,3,ug/m3
3,24f92792-dc63-4b2c-a715-69410ebd1f02,104,0b30092e-0bac-4a8a-a21b-ea3916ae6e85,0fc540c4-cae8-4557-a091-6e310dd8d095,0fc540c4-cae8-4557-a091-6e310dd8d095,4,m/s
4,f6e05409-ede2-45a9-b5d8-5d260f28af48,105,0b30092e-0bac-4a8a-a21b-ea3916ae6e85,85e37543-6901-475a-83e1-0647bbd4b2bc,85e37543-6901-475a-83e1-0647bbd4b2bc,5,deg


In [13]:
full_quantity_units= pd.merge(
    full_units, 
    quantity[['_id', 'code_quantity', 'shortname', 'longname', 'attrib']], 
    left_on='code_quantity._id', 
    right_on='_id', 
    how='left',
    suffixes=('', '_qty')
)

full_quantity_units = full_quantity_units.drop(["_id_x", "code_quantity._id", "code_unit._id", "_id_y", "_id"], axis=1)
full_quantity_units.head()

,code_combi,code_unit,unitname,code_quantity,shortname,longname,attrib
0,101,1,ppb,1,SO2,sulphur dioxide,gasep
1,102,2,ppm,1,SO2,sulphur dioxide,gasep
2,103,3,ug/m3,1,SO2,sulphur dioxide,gasep
3,104,4,m/s,1,SO2,sulphur dioxide,gasep
4,105,5,deg,1,SO2,sulphur dioxide,gasep


### Jungimas į vieną bendrą rinkinį

In [19]:
air_quality_data = pd.merge(
    averages_stations, 
    full_quantity_units, 
    left_on='code_combi', 
    right_on='code_combi', 
    how='left'
)
air_quality_data = air_quality_data.drop(["_id_x", "id", "code_combi", "code_unit", "code_quantity", "atribut"], axis=1)
air_quality_data.head()

,ldatetime,lvalue,stat_num,city,unitname,shortname,longname,attrib
0,2012-12-11T10:00:00,0.00,2,Vilnius,ppb,ISBTEN,iso-butene-PID,Oprek
1,2012-12-11T10:00:00,0.00,2,Vilnius,ppb,ISPREN,isoprene-PID,Oprek
2,2012-12-11T10:00:00,1.03,2,Vilnius,ppb,NHPTAN,n-heptane,Oprek
3,2012-12-11T10:00:00,0.01,2,Vilnius,ppb,NHXA1,n-hexane,Oprek
4,2012-12-11T10:00:00,0.30,2,Vilnius,ppb,NHXAN,n-hexane,Oprek
